In [1]:
import json
import time
import re
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

DEV_PATH   = "project/data/dev.jsonl"
TEST_PATH  = "project/data/test_30.jsonl"
TRAIN_PATH = "project/data/train.jsonl"

RUN_TEST = True  

SPLITS = {
    "dev": DEV_PATH,
    **({"test": TEST_PATH} if RUN_TEST else {})
}

Path("outputs").mkdir(exist_ok=True, parents=True)
Path("project/results").mkdir(exist_ok=True, parents=True)

SYSTEM_MSG = (
    "You are a cautious healthcare assistant. "
    "Return ONLY a valid JSON object with keys: short_answer, confidence_level, clinical_notes. "
    "confidence_level must be one of: high, medium, low. "
    "No markdown, no code fences, no extra text."
)

print("Setup OK")
print("Splits:", list(SPLITS.keys()))

Setup OK
Splits: ['dev', 'test']


In [2]:
import json, re

TRAIN_PATH = "project/data/train.jsonl"  # ajusta se necessário

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train = [json.loads(l) for l in f if l.strip()]

needle = "GLP-1"
cands = [(i, ex.get("instruction","")) for i, ex in enumerate(train) if needle.lower() in ex.get("instruction","").lower()]

print("Matches:", len(cands))
for i, instr in cands[:20]:
    print(f"[{i}] {instr}")

shot = train[266]
ONE_SHOT_Q = shot["instruction"]
ONE_SHOT_A = shot["response"]

print("ONE_SHOT_Q:", ONE_SHOT_Q)
print("ONE_SHOT_A keys:", ONE_SHOT_A.keys())




Matches: 16
[22] A patient on GLP-1 RA reports nausea in week 2. They ask: What is the difference between association and causation in longevity studies?
[27] A patient on GLP-1 RA reports nausea in week 2. They ask: What is anabolic resistance in older adults?
[38] If evidence is limited, say so. Do GLP-1 receptor agonists mimic caloric restriction?
[59] A patient on GLP-1 RA reports nausea in week 2. They ask: Do GLP-1 receptor agonists mimic caloric restriction?
[82] What’s the evidence-based answer to: Do GLP-1 receptor agonists mimic caloric restriction?
[83] A patient on GLP-1 RA reports nausea in week 2. They ask: How does sleep relate to aging and metabolic health?
[150] A patient on GLP-1 RA reports nausea in week 2. They ask: Does inhibiting mTOR extend lifespan in humans?
[152] A patient with family history of MTC asks: Do GLP-1 receptor agonists mimic caloric restriction?
[204] A patient on GLP-1 RA reports nausea in week 2. They ask: What is cellular senescence in aging bi

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)
model.eval()

gen_cfg = GenerationConfig(
    max_new_tokens=260,   # one-shot precisa de um pouco mais
    do_sample=False,
    temperature=0.0,
    use_cache=True,
)

print("Model loaded.")
print("CUDA:", torch.cuda.is_available())


`torch_dtype` is deprecated! Use `dtype` instead!
2026-01-19 12:43:41.672033: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768826621.693902     514 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768826621.704454     514 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-19 12:43:41.902756: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded.
CUDA: True


In [4]:
def load_questions(path: str):
    qs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            q = obj.get("question") or obj.get("instruction")
            if q and isinstance(q, str):
                qs.append(q.strip())
    return qs

def extract_any_json(text: str):
    start_positions = [m.start() for m in re.finditer(r"\{", text)]
    for start in start_positions:
        depth = 0
        for i in range(start, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    candidate = text[start:i+1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        break
    return None


In [5]:
def build_one_shot_user_content(question: str) -> str:
    # o modelo vê 1 exemplo e depois a pergunta alvo
    example_json = json.dumps(ONE_SHOT_A, ensure_ascii=False)
    return (
        "Follow the format shown in the example. Return ONLY JSON.\n\n"
        f"Example question: {ONE_SHOT_Q}\n"
        f"Example JSON: {example_json}\n\n"
        f"User question: {question}\n"
        "JSON:"
    )

def generate_one_shot(question: str):
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": build_one_shot_user_content(question)},
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # ajuda a começar JSON
    prompt = prompt + "{"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            generation_config=gen_cfg,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    dt = time.time() - t0

    full_text = tokenizer.decode(out[0], skip_special_tokens=True)
    prompt_text = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)
    gen_text = full_text[len(prompt_text):].strip()

    obj = extract_any_json(gen_text)
    if obj is None:
        obj = {
            "short_answer": "",
            "confidence_level": "low",
            "clinical_notes": "Model output did not contain valid JSON."
        }

    return obj, gen_text, dt


In [6]:
def run_split(split_name: str, in_path: str, method_name: str = "one"):
    questions = load_questions(in_path)
    n = len(questions)
    if n == 0:
        raise ValueError(f"Não carreguei perguntas de {in_path}")

    out_path = f"outputs/{split_name}_one_shot.jsonl"
    results = []

    print(f"\n=== RUN {method_name.upper()} | split={split_name} | n={n} ===")
    print("Input:", in_path)
    print("Output:", out_path)

    t_global0 = time.time()
    times = []

    for i, q in enumerate(questions, start=1):
        t_start = time.time()
        print(f"[{split_name}] {i}/{n} → starting at {time.strftime('%H:%M:%S')}", flush=True)

        obj, raw_text, dt = generate_one_shot(q)

        t_end = time.time()
        avg = (t_end - t_global0) / i
        eta = avg * (n - i)

        results.append({
            "split": split_name,
            "method": method_name,
            "question": q,
            "parsed_json": obj,
            "raw_text": raw_text,
            "timing_sec": dt
        })

    print(
        f"[{split_name}] {i}/{n} ← finished in {t_end - t_start:.2f}s | "
        f"avg {avg:.2f}s | ETA {eta/60:.1f} min",
        flush=True
    )


    with open(out_path, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    total = time.time() - t_global0
    avg_time = (sum(times) / len(times)) if len(times) > 0 else float("nan")
    print(f"Saved: {out_path} | total {total/60:.1f} min | avg {avg_time:.2f}s/question")


    return out_path

generated_files = []
for split_name, in_path in SPLITS.items():
    generated_files.append(run_split(split_name, in_path, method_name="one"))




=== RUN ONE | split=dev | n=30 ===
Input: project/data/dev.jsonl
Output: outputs/dev_one_shot.jsonl
[dev] 1/30 → starting at 12:45:41
[dev] 2/30 → starting at 12:45:45
[dev] 3/30 → starting at 12:45:48
[dev] 4/30 → starting at 12:45:51
[dev] 5/30 → starting at 12:45:53
[dev] 6/30 → starting at 12:45:56
[dev] 7/30 → starting at 12:46:00
[dev] 8/30 → starting at 12:46:02
[dev] 9/30 → starting at 12:46:05
[dev] 10/30 → starting at 12:46:08
[dev] 11/30 → starting at 12:46:12
[dev] 12/30 → starting at 12:46:14
[dev] 13/30 → starting at 12:46:18
[dev] 14/30 → starting at 12:46:20
[dev] 15/30 → starting at 12:46:24
[dev] 16/30 → starting at 12:46:27
[dev] 17/30 → starting at 12:46:30
[dev] 18/30 → starting at 12:46:33
[dev] 19/30 → starting at 12:46:35
[dev] 20/30 → starting at 12:46:38
[dev] 21/30 → starting at 12:46:41
[dev] 22/30 → starting at 12:46:44
[dev] 23/30 → starting at 12:46:47
[dev] 24/30 → starting at 12:46:50
[dev] 25/30 → starting at 12:46:52
[dev] 26/30 → starting at 12:46:5

In [7]:
import random

required = {"short_answer", "confidence_level", "clinical_notes"}
allowed_conf = {"high", "medium", "low"}

for path in generated_files:
    rows = [json.loads(l) for l in open(path, "r", encoding="utf-8") if l.strip()]
    print("\nFILE:", path, "| lines:", len(rows))

    missing = 0
    bad_conf = 0
    for r in rows:
        pj = r.get("parsed_json", {})
        if not isinstance(pj, dict) or not required.issubset(set(pj.keys())):
            missing += 1
        conf = str(pj.get("confidence_level", "")).strip().lower()
        if conf not in allowed_conf:
            bad_conf += 1

    print("Missing required keys:", missing)
    print("Bad confidence_level:", bad_conf)

    print("\nSample:")
    s = random.choice(rows)
    print("Q:", s["question"])
    print("Parsed:", s["parsed_json"])



FILE: outputs/dev_one_shot.jsonl | lines: 30
Missing required keys: 0
Bad confidence_level: 0

Sample:
Q: Answer conservatively and include safety notes if relevant: What is immunosenescence?
Parsed: {'short_answer': '', 'confidence_level': 'low', 'clinical_notes': 'Model output did not contain valid JSON.'}

FILE: outputs/test_one_shot.jsonl | lines: 30
Missing required keys: 0
Bad confidence_level: 0

Sample:
Q: Why is long-term safety data important for longevity interventions?
Parsed: {'short_answer': '', 'confidence_level': 'low', 'clinical_notes': 'Model output did not contain valid JSON.'}


In [8]:
import boto3
from pathlib import Path

S3_BUCKET = "healthcare-longevity"
S3_PREFIX = "healthcare-longevity"

s3 = boto3.client("s3")

def upload_if_exists(local_path: str, s3_key: str):
    p = Path(local_path)
    if not p.exists():
        print("SKIP (not found):", local_path)
        return
    s3.upload_file(str(p), S3_BUCKET, s3_key)
    print(f"UPLOADED → s3://{S3_BUCKET}/{s3_key}")

print("\n=== UPLOADING ONE-SHOT OUTPUTS ===")
upload_if_exists("outputs/dev_one_shot.jsonl",  f"{S3_PREFIX}/results/dev_one_shot.jsonl")
upload_if_exists("outputs/test_one_shot.jsonl", f"{S3_PREFIX}/results/test_one_shot.jsonl")

print("\n=== UPLOADING DATA + PROMPT (OPTIONAL) ===")
upload_if_exists("project/data/train-2.jsonl", f"{S3_PREFIX}/data/train-2.jsonl")
upload_if_exists("project/data/test_30.jsonl", f"{S3_PREFIX}/data/test_30.jsonl")

# Se quiseres guardar o “prompt” do one-shot, podemos escrever e subir também.
print("\nDone.")



=== UPLOADING ONE-SHOT OUTPUTS ===
UPLOADED → s3://healthcare-longevity/healthcare-longevity/results/dev_one_shot.jsonl
UPLOADED → s3://healthcare-longevity/healthcare-longevity/results/test_one_shot.jsonl

=== UPLOADING DATA + PROMPT (OPTIONAL) ===
SKIP (not found): project/data/train-2.jsonl
UPLOADED → s3://healthcare-longevity/healthcare-longevity/data/test_30.jsonl

Done.


In [9]:
import json, random

PATH = "outputs/dev_one_shot.jsonl"
rows = [json.loads(l) for l in open(PATH, "r", encoding="utf-8") if l.strip()]
r = random.choice(rows)

print("Q:", r["question"])
print("\nRAW_TEXT (first 800 chars):\n", r["raw_text"][:800])
print("\nPARSED_JSON:\n", r["parsed_json"])



Q: A patient with suspected gastroparesis asks: What is mTOR and why is it discussed in longevity research?

RAW_TEXT (first 800 chars):
 "short_answer": "mTOR is a cellular pathway that regulates cell growth, proliferation, and survival. In longevity research, it is discussed because inhibiting mTOR can extend lifespan in model organisms, suggesting potential anti-aging benefits.", "confidence_level": "medium", "clinical_notes": "While mTOR inhibition shows promise in extending lifespan in animal models, its direct application in human longevity is still under investigation and not yet clinically established." }

PARSED_JSON:
 {'short_answer': '', 'confidence_level': 'low', 'clinical_notes': 'Model output did not contain valid JSON.'}


In [10]:
import json
import re
from pathlib import Path

def extract_any_json(text: str):
    start_positions = [m.start() for m in re.finditer(r"\{", text)]
    for start in start_positions:
        depth = 0
        for i in range(start, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    candidate = text[start:i+1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        break
    return None

def fix_file(in_path: str, out_path: str):
    inp = Path(in_path)
    if not inp.exists():
        print("SKIP (not found):", in_path)
        return

    rows = [json.loads(l) for l in inp.open("r", encoding="utf-8") if l.strip()]
    fixed = 0

    with open(out_path, "w", encoding="utf-8") as f:
        for r in rows:
            text = (r.get("raw_text") or "").strip()
            if text.startswith('"'):
                text = "{" + text

            obj = extract_any_json(text)
            if obj is not None:
                r["parsed_json"] = obj
                fixed += 1

            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"Wrote: {out_path} | fixed {fixed}/{len(rows)}")

fix_file("outputs/dev_one_shot.jsonl",  "outputs/dev_one_shot_fixed.jsonl")
fix_file("outputs/test_one_shot.jsonl", "outputs/test_one_shot_fixed.jsonl")


Wrote: outputs/dev_one_shot_fixed.jsonl | fixed 30/30
Wrote: outputs/test_one_shot_fixed.jsonl | fixed 30/30
